#### 清理之前的安装
pip uninstall pydantic pydantic-core langchain langchain-core langchain-community -y

##### 安装兼容的版本组合
pip install pydantic==2.10.3
pip install langchain==0.3.15
pip install langchain-core==0.3.28
pip install langchain-community==0.3.14
pip install dashscope==1.20.11
pip install requests==2.32.3

In [3]:
import os
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
import requests
from dotenv import load_dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()
DASHSCOPE_API_KEY= os.getenv("DASHSCOPE_API_KEY")
print(DASHSCOPE_API_KEY)


sk-832e922df78542388c19198ea123bda7


In [5]:
@tool
def get_weather(city: str) -> str:
    """
    查询指定城市的当前天气信息
    
    Args:
        city: 城市名称，支持中文和英文，例如 "北京", "Beijing", "上海", "Shanghai"
    
    Returns:
        包含天气信息的字符串
    """
    try:
        # 使用wttr.in免费天气API（无需注册）
        # 支持中文城市名
        url = f"https://wttr.in/{city}?format=j1&lang=zh"
        
        print(f"🔍 正在查询: {city}")
        
        response = requests.get(url, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            
            # 提取当前天气信息
            current = data['current_condition'][0]
            temp_c = current['temp_C']
            feels_like = current['FeelsLikeC']
            humidity = current['humidity']
            weather_desc = current['lang_zh'][0]['value'] if current.get('lang_zh') else current['weatherDesc'][0]['value']
            wind_speed = current['windspeedKmph']
            
            # 格式化返回结果
            weather_info = f"""
城市: {city}
当前温度: {temp_c}°C
体感温度: {feels_like}°C
天气状况: {weather_desc}
湿度: {humidity}%
风速: {wind_speed} km/h
"""
            return weather_info.strip()
        else:
            return f"无法获取{city}的天气信息，请检查城市名称是否正确"
            
    except requests.exceptions.Timeout:
        return f"查询{city}天气超时，请稍后重试"
    except Exception as e:
        return f"查询天气时出错: {str(e)}"

In [6]:
@tool
def get_weather_forecast(city: str) -> str:
    """
    查询指定城市未来3天的天气预报
    
    Args:
        city: 城市名称，支持中文和英文
    
    Returns:
        未来3天的天气预报信息
    """
    try:
        url = f"https://wttr.in/{city}?format=j1&lang=zh"
        
        print(f"🔍 正在查询{city}的天气预报")
        
        response = requests.get(url, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            
            forecast_info = f"{city}未来3天天气预报:\n\n"
            
            # 获取未来3天的预报
            for day_data in data['weather'][:3]:
                date = day_data['date']
                max_temp = day_data['maxtempC']
                min_temp = day_data['mintempC']
                desc = day_data['lang_zh'][0]['value'] if day_data.get('lang_zh') else day_data['hourly'][0]['weatherDesc'][0]['value']
                
                forecast_info += f"日期: {date}\n"
                forecast_info += f"温度: {min_temp}°C ~ {max_temp}°C\n"
                forecast_info += f"天气: {desc}\n\n"
            
            return forecast_info.strip()
        else:
            return f"无法获取{city}的天气预报"
            
    except Exception as e:
        return f"查询天气预报时出错: {str(e)}"

In [7]:
@tool  
def compare_weather(city1: str, city2: str) -> str:
    """
    比较两个城市的天气情况
    
    Args:
        city1: 第一个城市名称
        city2: 第二个城市名称
    
    Returns:
        两个城市的天气对比信息
    """
    try:
        # 获取两个城市的天气
        weather1_data = requests.get(f"https://wttr.in/{city1}?format=j1&lang=zh", timeout=10).json()
        weather2_data = requests.get(f"https://wttr.in/{city2}?format=j1&lang=zh", timeout=10).json()
        
        temp1 = int(weather1_data['current_condition'][0]['temp_C'])
        temp2 = int(weather2_data['current_condition'][0]['temp_C'])
        
        desc1 = weather1_data['current_condition'][0].get('lang_zh', [{}])[0].get('value', '未知')
        desc2 = weather2_data['current_condition'][0].get('lang_zh', [{}])[0].get('value', '未知')
        
        comparison = f"""
天气对比:

{city1}:
  温度: {temp1}°C
  天气: {desc1}

{city2}:
  温度: {temp2}°C  
  天气: {desc2}

温差: {abs(temp1 - temp2)}°C
"""
        
        if temp1 > temp2:
            comparison += f"\n{city1}比{city2}热 {temp1 - temp2}°C"
        elif temp2 > temp1:
            comparison += f"\n{city2}比{city1}热 {temp2 - temp1}°C"
        else:
            comparison += f"\n两个城市温度相同"
            
        return comparison.strip()
        
    except Exception as e:
        return f"比较天气时出错: {str(e)}"

In [8]:
def create_weather_agent():
    """创建天气查询Agent"""
    
    # 初始化千问大模型
    llm = ChatTongyi(
        model="qwen-plus",
        temperature=0.7,
    )
    
    # 定义工具列表
    tools = [get_weather, get_weather_forecast, compare_weather]
    
    # 创建提示词模板
    prompt = ChatPromptTemplate.from_messages([
        ("system", """你是一个专业的天气助手。你可以使用以下工具:

1. get_weather: 查询指定城市的当前天气
2. get_weather_forecast: 查询指定城市未来3天的天气预报
3. compare_weather: 比较两个城市的天气情况

当用户询问天气时，请:
- 识别用户想查询的城市名称（支持中文和英文）
- 选择合适的工具进行查询
- 用友好、自然的语言回答用户
- 如果用户问哪个城市更热/更冷，使用compare_weather工具

请始终使用中文回答。"""),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ])
    
    # 创建Agent
    agent = create_tool_calling_agent(llm, tools, prompt)
    
    # 创建AgentExecutor
    agent_executor = AgentExecutor(
        agent=agent,
        tools=tools,
        verbose=True,
        handle_parsing_errors=True,
        max_iterations=5,
    )
    
    return agent_executor

In [9]:
agent = create_weather_agent()

In [10]:
result = agent.invoke({"input": "北京今天天气怎么样?"})
print(f"\n【回答】\n{result['output']}\n")



> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': '北京'}`


🔍 正在查询: 北京
城市: 北京
当前温度: 16°C
体感温度: 16°C
天气状况: 晴天
湿度: 17%
风速: 6 km/h北京今天天气晴朗，当前温度为16°C，体感温度也是16°C。空气比较干燥，湿度只有17%，有轻微的风，风速为6公里/小时。适合外出活动，但早晚温差可能较大，建议适当增减衣物。

> Finished chain.

【回答】
北京今天天气晴朗，当前温度为16°C，体感温度也是16°C。空气比较干燥，湿度只有17%，有轻微的风，风速为6公里/小时。适合外出活动，但早晚温差可能较大，建议适当增减衣物。

